In [0]:
%pip install requests

In [0]:
import requests
import json
from datetime import datetime


In [0]:
dbutils.widgets.text("job_name", "Medallion Architecture", "Nome del Job")
dbutils.widgets.text("job_status",  "SUCCESS", "Stato del Job")
dbutils.widgets.text("run_url", "https://dbc-ffab9e1e-aa5d.cloud.databricks.com", "URL del Run")

job_name = dbutils.widgets.get("job_name")
job_status = dbutils.widgets.get("job_status")
run_url = dbutils.widgets.get("run_url")


In [0]:
def send_slack_notification(status: str, job_name: str, run_url: str = "", details: str = ""):
    webhook_url = dbutils.secrets.get(scope = "monitoring", key = "slack_webhook")
    status = status.upper()
    
    color = "#36a64f" if status == "SUCCESS" else "#e01e5a"
    icon = "✅" if status == "SUCCESS" else "❌"

    blocks = [
        {
            "type": "section",
            "text": {
                "type": "mrkdwn",
                "text": f"{icon} *{job_name}* — `{status}`"
            }
        },
        {
            "type": "section",
            "fields": [
                {
                    "type": "mrkdwn",
                    "text": f"*Timestamp:*\n{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
                },
                {
                    "type": "mrkdwn",
                    "text": f"*Details:*\n{details if details else 'No issues'}"
                }
            ]
        }
    ]

    if run_url and run_url != "https://dbc-ffab9e1e-aa5d.cloud.databricks.com":
        blocks.append(
            
            {
                "type": "section",
                "text": {
                    "type": "mrkdwn",
                    "text": f"<{run_url}| Apri la Run su Databricks>"
                }
            }

        )

    payload = {"attachments": [ {"color": color, "blocks": blocks} ] }

    response = requests.post(webhook_url, data = json.dumps(payload) )

    print(f"Response: {response}")


In [0]:
try:
    #Conta i record nei vari layer
    count_bronze = spark.table("notebook_breweries.bronze_breweries").count()
    count_silver = spark.table("notebook_breweries.silver_breweries").count()
    #count_gold_current = spark.table("notebook_breweries.gold_breweries").filter("current = true").count()
    count_gold = spark.table("notebook_breweries.gold_breweries").count()
    
    details = (
        f"🥉 Bronze: {count_bronze} records\n"
        f"🥈 Silver: {count_silver} records\n"
        f"🥇 Gold: {count_gold} records"        
    )

    send_slack_notification(
        status = job_status, 
        job_name = job_name, 
        run_url = run_url,
        details = details
        )
    
except Exception as e:
    send_slack_notification(
        status = "failed",
        job_name = job_name,
        run_url = run_url,
        details = str(e)
    )
    raise